In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader
from tqdm import tqdm
import math
import os

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

In [ ]:
data_dir = "/content/dataset"   # <------ CHANGE THIS

In [ ]:
transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

train_dataset = datasets.ImageFolder(root=data_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)

num_classes = len(train_dataset.classes)
print("Classes:", num_classes)

In [ ]:
class FaceEmbeddingNet(nn.Module):
    def __init__(self, embedding_size=512, pretrained=True):
        super().__init__()
        self.model = models.resnet50(pretrained=pretrained)
        self.model.fc = nn.Identity()
        self.embedding = nn.Linear(2048, embedding_size)
        nn.init.xavier_normal_(self.embedding.weight)

    def forward(self, x):
        x = self.model(x)
        x = self.embedding(x)
        return F.normalize(x, p=2, dim=1)

In [ ]:
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, scale=64.0, margin=0.5):
        super().__init__()
        self.s = scale
        self.m = margin

        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.th = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin

    def forward(self, embeddings, labels):
        W = F.normalize(self.weight, p=2, dim=1)
        cos = F.linear(embeddings, W).clamp(-1, 1)
        sin = torch.sqrt(1 - cos**2)

        cos_m = cos * self.cos_m - sin * self.sin_m

        one_hot = torch.zeros_like(cos)
        one_hot.scatter_(1, labels.view(-1,1), 1)

        output = one_hot * cos_m + (1 - one_hot) * cos
        output *= self.s
        return output

In [ ]:
class ArcFaceModel(nn.Module):
    def __init__(self, num_classes, embedding_size=512):
        super().__init__()
        self.backbone = FaceEmbeddingNet(embedding_size)
        self.arc = ArcMarginProduct(embedding_size, num_classes)

    def forward(self, x, labels):
        emb = self.backbone(x)
        logits = self.arc(emb, labels)
        return emb, logits

In [ ]:
model = ArcFaceModel(num_classes=num_classes).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for imgs, labels in progress:
        imgs, labels = imgs.to(device), labels.to(device)

        emb, logits = model(imgs, labels)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1}: Avg Loss = {total_loss/len(train_loader):.4f}")

In [ ]:
os.makedirs("checkpoints", exist_ok=True)
torch.save(model.state_dict(), "checkpoints/arcface.pth")

print("Model saved!")

In [ ]:
@torch.no_grad()
def get_embedding(model, img_tensor):
    model.eval()
    img_tensor = img_tensor.unsqueeze(0).to(device)
    emb = model.backbone(img_tensor)
    return emb.cpu()

def cosine_similarity(v1, v2):
    return F.cosine_similarity(v1, v2).item()

In [ ]:
img1 = transform(Image.open("test1.jpg"))
img2 = transform(Image.open("test2.jpg"))

emb1 = get_embedding(model, img1)
emb2 = get_embedding(model, img2)

print("Similarity:", cosine_similarity(emb1, emb2))